In [ ]:
!pip uninstall -y torchao -q
!pip install -q "numpy<2"
!pip install -q torch==2.2.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q "torchdata==0.7.1"

In [ ]:
!pip uninstall -y dgl -q
!pip install -q dgl -f https://data.dgl.ai/wheels/cu121/repo.html

In [ ]:
!ls /usr/local/lib/python3.12/dist-packages/dgl/graphbolt/ | grep libgraphbolt

In [ ]:
!pip uninstall -y torchvision torchaudio -q

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
# Pin `alignn` to the exact release that produced alignn_encoder.pt. ALIGNNWithEmbedding
# below overrides ALIGNN.forward with a hard-coded copy of that version's internals, and
# the signature has already changed once across releases (older builds unpack
# `g, lg, lat` instead of `g, lg`). Read the version out of alignn_encoder_meta.json,
# written by Band_ALIGNN_Encoder.ipynb, and put it here.
ALIGNN_PIN = ""  # e.g. "==2024.5.27"

!pip install -q "transformers<5" "alignn{ALIGNN_PIN}" huggingface_hub pandas scikit-learn tqdm
!pip uninstall -y torchao -q


In [ ]:
import numpy as np, torch, dgl, transformers, alignn
print(np.__version__, torch.__version__, torch.cuda.is_available(), dgl.__version__, transformers.__version__)
print("alignn:", getattr(alignn, "__version__", "unknown"))

g = dgl.graph(([0, 1], [1, 2])).to("cuda")
print("dgl graph device:", g.device)

from transformers import AutoModel
m = AutoModel.from_pretrained("m3rg-iitd/matscibert")
print("MatSciBERT loaded OK")


In [ ]:
import os, json, random, hashlib
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from collections import OrderedDict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW
import dgl
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup
from alignn.models.alignn import ALIGNN, ALIGNNConfig

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

REPO_ID = "Godseye1311/alignn-band-gap"
LOCAL_DIR = "./mmfuse_data"
TRANSFORMER_NAME = "m3rg-iitd/matscibert"

# --- pretrained graph encoder -------------------------------------------------
# Algorithm 1 (appendix G) requires a *pretrained* GNN alongside the pretrained
# transformer. MatSciBERT arrives pretrained via from_pretrained(); ALIGNN does not, so
# it has to be loaded explicitly from the band-gap checkpoint trained in
# Band_ALIGNN_Encoder.ipynb. Without this the graph branch starts from random weights at
# lr=1e-5 and the model is effectively a MatSciBERT-only regressor.
ALIGNN_CKPT_FILE = "alignn_encoder.pt"   # filename inside REPO_ID
ALIGNN_META_FILE = "alignn_encoder_meta.json"

# The encoder's own train/val/test split, reproduced below so this model does not
# validate or test on materials the encoder was trained on.
ENCODER_SPLIT_SEED = 0     # SEED in Band_ALIGNN_Encoder.ipynb
VAL_FRACTION = 0.1
TEST_FRACTION = 0.1

# --- reproducibility ----------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


In [ ]:
from huggingface_hub import snapshot_download

local_path = snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    local_dir=LOCAL_DIR,
    ignore_patterns=["xrd_data.h5"],  # not used here
)
print("Downloaded to:", local_path)

In [ ]:
text_data_path = os.path.join(LOCAL_DIR, "text_data.csv")
tabular_path = os.path.join(LOCAL_DIR, "materials_tabular.csv")

text_df = pd.read_csv(text_data_path)
tabular_df = pd.read_csv(tabular_path)

print("text_data.csv columns:", list(text_df.columns))
print(text_df.head(3))
print("materials_tabular.csv columns:", list(tabular_df.columns))
print(tabular_df.head(3))

In [ ]:
ID_COL = "material_id"       # placeholder — set to real ID column
TEXT_COL = "text"            # placeholder — set to real text column
TARGET_COL = "band_gap"      # placeholder — set to real target column
TARGET_FILE = "tabular"      # "tabular" or "text", wherever TARGET_COL lives

assert ID_COL in text_df.columns
target_source_df = tabular_df if TARGET_FILE == "tabular" else text_df
assert TARGET_COL in target_source_df.columns

In [ ]:
ID_COL = "material_id"
TEXT_COL = "description"
TARGET_COL = "band_gap"
TARGET_FILE = "tabular"

assert ID_COL in text_df.columns
target_source_df = tabular_df if TARGET_FILE == "tabular" else text_df
assert TARGET_COL in target_source_df.columns

In [ ]:
with open(os.path.join(LOCAL_DIR, "alignn_graphs", "graph_shards.json")) as f:
    graph_index = json.load(f)

print(f"Graphs available: {len(graph_index)}")

merged = text_df[[ID_COL, TEXT_COL]].merge(
    tabular_df[[ID_COL, TARGET_COL]], on=ID_COL, how="inner"
)
merged = merged.dropna(subset=[TEXT_COL, TARGET_COL])
merged = merged[merged[ID_COL].astype(str).isin(graph_index.keys())]
merged = merged.drop_duplicates(subset=[ID_COL]).reset_index(drop=True)
print(f"Rows with text + target + graph: {len(merged)}")

# ---- reuse the ALIGNN encoder's split ----------------------------------------
# The pretrained ALIGNN was trained on 80% of these materials. Drawing an independent
# split here (the previous sklearn train_test_split(random_state=42)) would put most of
# the encoder's training materials into this model's validation and test sets, so the
# reported MAE would be measured on data the graph branch has already memorised.
#
# Reconstruct the encoder notebook's split exactly: same CSV, same shard filter, same
# seeded permutation, same fractions. Verify the printed signature matches the one
# Band_ALIGNN_Encoder.ipynb prints; if it does not, the two notebooks disagree and the
# split must be loaded from alignn_split.json instead.
_enc_df = tabular_df[tabular_df[ID_COL].isin(graph_index.keys())].reset_index(drop=True)
_enc_ids = _enc_df[ID_COL].astype(str).tolist()
_n = len(_enc_ids)
_perm = np.random.RandomState(ENCODER_SPLIT_SEED).permutation(_n)
_n_val, _n_test = int(VAL_FRACTION * _n), int(TEST_FRACTION * _n)
_n_train = _n - _n_val - _n_test

split_of = {}
for positions, name in (
    (_perm[:_n_train], "train"),
    (_perm[_n_train:_n_train + _n_val], "val"),
    (_perm[_n_train + _n_val:], "test"),
):
    for i in positions:
        split_of[_enc_ids[i]] = name

_split_file = os.path.join(LOCAL_DIR, "alignn_split.json")
if os.path.exists(_split_file):
    with open(_split_file) as f:
        _saved = json.load(f)
    assert _saved == split_of, "reconstructed split disagrees with alignn_split.json"
    print("Split matches alignn_split.json from the encoder run.")

_sig = hashlib.sha1("|".join(f"{m}:{split_of[m]}" for m in sorted(split_of)).encode()).hexdigest()[:12]
print(f"Encoder split reconstructed over {_n:,} materials  |  signature {_sig}")
print("^ this must equal the signature printed by Band_ALIGNN_Encoder.ipynb")

merged["split"] = merged[ID_COL].astype(str).map(split_of)
assert merged["split"].notna().all(), "some materials are absent from the encoder split"
print(merged["split"].value_counts().to_dict())


In [ ]:
# Number of shards the sampler mixes together before emitting batches (see the sampler
# below). The train cache has to hold a whole buffer resident, plus a little slack.
BUFFER_SHARDS = 12
MAX_CACHED_SHARDS_TRAIN = BUFFER_SHARDS + 2
MAX_CACHED_SHARDS_EVAL = 4

_shard_cache_train = OrderedDict()
_shard_cache_eval = OrderedDict()

def _load_shard(shard_name, cache, max_size):
    if shard_name in cache:
        cache.move_to_end(shard_name)
        return cache[shard_name]
    atom_path = os.path.join(LOCAL_DIR, "alignn_graphs", f"{shard_name}_atom.bin")
    line_path = os.path.join(LOCAL_DIR, "alignn_graphs", f"{shard_name}_line.bin")
    atom_graphs, _ = dgl.load_graphs(atom_path)
    line_graphs, _ = dgl.load_graphs(line_path)
    cache[shard_name] = (atom_graphs, line_graphs)
    if len(cache) > max_size:
        cache.popitem(last=False)
    return cache[shard_name]

def get_graph_pair(material_id, is_eval=False):
    entry = graph_index[str(material_id)]
    shard_name, idx = entry["shard"], entry["index"]
    if is_eval:
        atom_graphs, line_graphs = _load_shard(shard_name, _shard_cache_eval, MAX_CACHED_SHARDS_EVAL)
    else:
        atom_graphs, line_graphs = _load_shard(shard_name, _shard_cache_train, MAX_CACHED_SHARDS_TRAIN)
    return atom_graphs[idx], line_graphs[idx]


In [ ]:
from torch.utils.data import Sampler
import random


class ShardBufferedSampler(Sampler):
    """Shard-aware sampler that still produces well-mixed batches.

    The previous ShardGroupedSampler emitted one shard at a time, so every batch of 32
    came from a single shard. Shard order tracks material_id order, which is not
    arbitrary, so batches were chemically correlated rather than i.i.d. That is bad in
    general and specifically bad here: ALIGNN is full of BatchNorm1d layers whose
    statistics are estimated per batch, and homogeneous batches give them systematically
    wrong estimates.

    This version shuffles across a buffer of `buffer_shards` shards at a time, so a batch
    mixes roughly `buffer_shards x shard_size` materials while the LRU cache still only
    has to hold `buffer_shards` shards resident.
    """

    def __init__(self, dataset, buffer_shards, shuffle=True, seed=SEED):
        self.dataset = dataset
        self.buffer_shards = buffer_shards
        self.shuffle = shuffle
        self.seed = seed
        self.epoch = 0
        self.shard_to_indices = {}
        for i, mid in enumerate(dataset.ids):
            self.shard_to_indices.setdefault(graph_index[mid]["shard"], []).append(i)

    def set_epoch(self, epoch):
        """Reshuffle differently each epoch (call from the training loop)."""
        self.epoch = epoch

    def __iter__(self):
        shard_names = sorted(self.shard_to_indices)
        rng = random.Random(self.seed + self.epoch)
        if self.shuffle:
            rng.shuffle(shard_names)
        for start in range(0, len(shard_names), self.buffer_shards):
            block = shard_names[start:start + self.buffer_shards]
            idxs = [i for s in block for i in self.shard_to_indices[s]]
            if self.shuffle:
                rng.shuffle(idxs)
            yield from idxs

    def __len__(self):
        return len(self.dataset)


In [ ]:
class ALIGNNTextDataset(Dataset):
    def __init__(self, df, is_eval=False):
        self.ids = df[ID_COL].astype(str).tolist()
        self.texts = df[TEXT_COL].tolist()
        self.labels = df[TARGET_COL].astype(float).tolist()
        self.is_eval = is_eval

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        g, lg = get_graph_pair(self.ids[idx], is_eval=self.is_eval)
        return self.texts[idx], (g, lg), self.labels[idx]

def collate_fn(batch):
    texts, graph_pairs, labels = zip(*batch)
    gs, lgs = zip(*graph_pairs)
    batched_g = dgl.batch(gs)
    batched_lg = dgl.batch(lgs)
    labels = torch.tensor(labels, dtype=torch.float32)
    return list(texts), (batched_g, batched_lg), labels

BATCH_SIZE = 32

# Splits come from the encoder split reconstructed above, not from a fresh
# train_test_split -- see the leakage note there.
train_df = merged[merged["split"] == "train"].reset_index(drop=True)
val_df = merged[merged["split"] == "val"].reset_index(drop=True)
test_df = merged[merged["split"] == "test"].reset_index(drop=True)
print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")

train_dataset = ALIGNNTextDataset(train_df, is_eval=False)
val_dataset = ALIGNNTextDataset(val_df, is_eval=True)
test_dataset = ALIGNNTextDataset(test_df, is_eval=True)

train_sampler = ShardBufferedSampler(train_dataset, BUFFER_SHARDS, shuffle=True)
val_sampler = ShardBufferedSampler(val_dataset, BUFFER_SHARDS, shuffle=False)
test_sampler = ShardBufferedSampler(test_dataset, BUFFER_SHARDS, shuffle=False)

# drop_last on train: a final batch of size 1 makes ALIGNN's BatchNorm1d layers fail
# outright in train mode (batch variance is undefined for a single sample).
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=train_sampler,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True,
                          persistent_workers=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, sampler=val_sampler, collate_fn=collate_fn, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, sampler=test_sampler, collate_fn=collate_fn, num_workers=0)


In [ ]:
_sample_id = merged[ID_COL].iloc[0]
_g, _lg = get_graph_pair(_sample_id, is_eval=True)
print("atom graph ndata:", list(_g.ndata.keys()))
print("atom graph edata:", list(_g.edata.keys()))
print("line graph edata:", list(_lg.edata.keys()))

assert "atom_features" in _g.ndata, "cached graphs are missing atom_features"
assert "r" in _g.edata, "cached graphs are missing edge vectors 'r'"
assert "h" in _lg.edata, "cached line graphs are missing angle features 'h'"
print("Atom feature dim:", int(_g.ndata["atom_features"].shape[-1]))


In [ ]:
class ALIGNNWithEmbedding(ALIGNN):
    """ALIGNN that also returns the pooled graph embedding `h` used for fusion.

    This is a copy of alignn.models.alignn.ALIGNN.forward with the readout exposed. It
    depends on the installed alignn release's module layout (angle_embedding,
    atom_embedding, edge_embedding, alignn_layers, gcn_layers, readout, fc) and on its
    input convention. Newer releases pass `[g, lg, lat]` and ignore `lat`; this override
    takes `(g, lg)`. Pin alignn via ALIGNN_PIN above to the version recorded in
    alignn_encoder_meta.json so this stays in sync with the checkpoint.
    """

    def forward(self, g):
        if len(g) == 3:          # tolerate the [g, lg, lat] convention; lat is unused
            g, lg, _ = g
        else:
            g, lg = g
        lg = lg.local_var()
        z = self.angle_embedding(lg.edata.pop("h"))
        g = g.local_var()
        x = self.atom_embedding(g.ndata.pop("atom_features"))
        bondlength = torch.norm(g.edata.pop("r"), dim=1)
        y = self.edge_embedding(bondlength)
        for alignn_layer in self.alignn_layers:
            x, y, z = alignn_layer(g, lg, x, y, z)
        for gcn_layer in self.gcn_layers:
            x, y = gcn_layer(g, x, y)
        h = self.readout(g, x)
        out = self.fc(h)
        if self.link:
            out = self.link(out)
        return out.view(-1), h


HIDDEN_FEATURES = 256
ATOM_INPUT_FEATURES = int(_g.ndata["atom_features"].shape[-1])

# Must match Band_ALIGNN_Encoder.ipynb's ALIGNNBandGap config exactly, or the checkpoint
# will not load. That notebook left atom_input_features at the alignn default (92); the
# assertion in the loading cell below catches any disagreement.
alignn_config = ALIGNNConfig(
    name="alignn",
    atom_input_features=ATOM_INPUT_FEATURES,
    hidden_features=HIDDEN_FEATURES,
    output_features=1,
    alignn_layers=4,
    gcn_layers=4,
    link="identity",
    classification=False,
)


In [ ]:
class ImprovedAttentionCombiner(nn.Module):
    """Multi-head cross attention: text embedding queries the structure embedding.

    Caveat, inherited from the paper's implementation and kept here for fidelity: both
    inputs are pooled to a single vector before this module, so the sequence length is 1
    and the softmax is over a single element -- the attention weights are identically
    1.0 and the module reduces to LayerNorm(W_o.W_v.h_s + h_t). Real, interpretable
    cross attention would require keeping MatSciBERT's token sequence as the query
    instead of mean-pooling it. Do not report attention weights from this module.
    """

    def __init__(self, dim, num_heads=8, dropout=0.2):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        assert self.head_dim * num_heads == dim
        self.query = nn.Linear(dim, dim)
        self.key = nn.Linear(dim, dim)
        self.value = nn.Linear(dim, dim)
        self.fc_out = nn.Linear(dim, dim)
        self.softmax = nn.Softmax(dim=-1)
        self.dropout = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, supervised_embedding, transformer_embedding):
        batch_size = supervised_embedding.size(0)
        supervised_embedding = self.norm1(supervised_embedding)
        transformer_embedding = self.norm1(transformer_embedding)
        query = self.query(transformer_embedding).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        key = self.key(supervised_embedding).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        value = self.value(supervised_embedding).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        attention_scores = torch.matmul(query, key.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attention_weights = self.dropout(self.softmax(attention_scores))
        combined = torch.matmul(attention_weights, value)
        combined = combined.transpose(1, 2).contiguous().view(batch_size, -1, self.dim)
        combined = self.fc_out(combined)
        combined = self.norm2(combined + transformer_embedding)
        return combined.squeeze(1)


class CombinedEmbeddingModel(nn.Module):
    def __init__(self, transformer_name, supervised_model, supervised_dim, combined_dim=512):
        super().__init__()
        self.device = device
        self.transformer = AutoModel.from_pretrained(transformer_name).to(self.device)
        self.supervised_model = supervised_model.to(self.device)
        self.supervised_proj = nn.Linear(supervised_dim, combined_dim).to(self.device)
        self.transformer_proj = nn.Linear(self.transformer.config.hidden_size, combined_dim).to(self.device)
        self.attention_combiner = ImprovedAttentionCombiner(combined_dim).to(self.device)
        self.fc = nn.Linear(combined_dim, 1).to(self.device)

    def forward(self, text, graph_pair, tokenizer):
        # MatSciBERT is capped at 512 tokens; table A2 gives a mean description length of
        # 741 words, so most descriptions are truncated. Unavoidable for BERT-family
        # encoders, but it should be stated when reporting results.
        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512).to(self.device)
        transformer_output = self.transformer(**inputs).last_hidden_state.mean(dim=1)
        transformer_proj = self.transformer_proj(transformer_output)

        g, lg = graph_pair
        g, lg = g.to(self.device), lg.to(self.device)
        # ALIGNN's backward pass is numerically fragile under reduced precision (its many
        # BatchNorm1d layers divide by variance); Band_ALIGNN_Encoder.ipynb confirmed NaN
        # gradients under autocast and forced fp32. Do the same here rather than relying
        # on bf16's wider exponent and skipping NaN batches after the fact.
        with torch.autocast(device_type="cuda", enabled=False):
            _, supervised_embedding = self.supervised_model((g, lg))
        supervised_proj = self.supervised_proj(supervised_embedding.float())

        combined_embedding = self.attention_combiner(supervised_proj.unsqueeze(1), transformer_proj.unsqueeze(1))
        return self.fc(combined_embedding)


In [ ]:
from huggingface_hub import hf_hub_download

tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_NAME)
alignn_encoder = ALIGNNWithEmbedding(alignn_config)

# --- load the pretrained ALIGNN band-gap encoder ------------------------------
def _fetch(filename):
    local = os.path.join(LOCAL_DIR, filename)
    if os.path.exists(local):
        return local
    return hf_hub_download(repo_id=REPO_ID, repo_type="dataset", filename=filename)

try:
    with open(_fetch(ALIGNN_META_FILE)) as f:
        enc_meta = json.load(f)
    print("encoder meta:", enc_meta)
    if getattr(alignn, "__version__", None) and enc_meta.get("alignn") not in (None, "unknown"):
        if alignn.__version__ != enc_meta["alignn"]:
            print(f"WARNING: alignn {alignn.__version__} installed but the checkpoint was "
                  f"produced with {enc_meta['alignn']} -- set ALIGNN_PIN and rerun.")
except Exception as e:  # meta file is optional
    enc_meta = {}
    print(f"({ALIGNN_META_FILE} unavailable: {e})")

state = torch.load(_fetch(ALIGNN_CKPT_FILE), map_location="cpu")
if isinstance(state, dict) and "model_state_dict" in state:
    state = state["model_state_dict"]
# Saved from ALIGNNBandGap, which wraps the ALIGNN backbone as self.backbone.
state = {k[len("backbone."):] if k.startswith("backbone.") else k: v for k, v in state.items()}

ckpt_in = int(state["atom_embedding.layer.0.weight"].shape[1])
ckpt_hidden = int(state["atom_embedding.layer.0.weight"].shape[0])
assert ckpt_in == ATOM_INPUT_FEATURES, (
    f"checkpoint expects atom_input_features={ckpt_in} but the cached graphs provide "
    f"{ATOM_INPUT_FEATURES}")
assert ckpt_hidden == HIDDEN_FEATURES, (
    f"checkpoint hidden_features={ckpt_hidden} but this config uses {HIDDEN_FEATURES}")

alignn_encoder.load_state_dict(state, strict=True)   # raises loudly on any layout drift
print(f"Loaded pretrained ALIGNN encoder ({len(state)} tensors, "
      f"atom_input_features={ckpt_in}, hidden={ckpt_hidden})")

model = CombinedEmbeddingModel(TRANSFORMER_NAME, alignn_encoder, supervised_dim=HIDDEN_FEATURES)


In [ ]:
from tqdm.auto import tqdm
from sklearn.metrics import mean_absolute_error, r2_score
import datetime

os.makedirs("./Results/Checkpoint", exist_ok=True)

# Figure B1 shows the paper's model converging within ~6-8 epochs, and figure B7(b) shows
# it overfitting past ~10 once the transformer is unfrozen. The previous num_epochs=100
# also stretched the cosine schedule's 10% warmup across 10 full epochs, so the learning
# rate was still ramping through the entire window in which the paper had converged.
num_epochs = 15
EARLY_STOP_PATIENCE = 4

# Appendix B.5 selects 1e-5. Kept as a single rate for all three branches to stay
# faithful to the paper; the fusion head (projections + attention + output) is randomly
# initialised, so if it visibly underfits, raise HEAD_LR to 1e-4 and report the change.
TEXT_LR, GRAPH_LR, HEAD_LR = 1e-5, 1e-5, 1e-5
WEIGHT_DECAY = 0.01

# Section 3: "The implementation uses SmoothL1Loss." Train, validation and test now all
# use it -- previously training minimised MSE while the test cell reported SmoothL1.
criterion = nn.SmoothL1Loss()

# Band gap has mean 0.874 / sigma 1.514 and is heavily zero-inflated (table A1). The
# paper does not standardise targets, so this stays off by default; flip it to True if
# you want to report the normalised variant, and say so in the writeup.
NORMALIZE_TARGETS = False
TARGET_MEAN = float(train_df[TARGET_COL].mean()) if NORMALIZE_TARGETS else 0.0
TARGET_STD = float(train_df[TARGET_COL].std()) if NORMALIZE_TARGETS else 1.0
print(f"target normalisation: {NORMALIZE_TARGETS} (mean={TARGET_MEAN:.4f}, std={TARGET_STD:.4f})")


def build_param_groups():
    """Per-branch learning rates, and no weight decay on biases / normalisation scales
    (1-D parameters) -- decaying those is the standard BERT fine-tuning mistake."""
    groups = []
    for module, lr in ((model.transformer, TEXT_LR), (model.supervised_model, GRAPH_LR)):
        params = [p for p in module.parameters() if p.requires_grad]
        groups.append({"params": [p for p in params if p.ndim > 1], "lr": lr, "weight_decay": WEIGHT_DECAY})
        groups.append({"params": [p for p in params if p.ndim <= 1], "lr": lr, "weight_decay": 0.0})
    head = [model.supervised_proj, model.transformer_proj, model.attention_combiner, model.fc]
    head_params = [p for m in head for p in m.parameters() if p.requires_grad]
    groups.append({"params": [p for p in head_params if p.ndim > 1], "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY})
    groups.append({"params": [p for p in head_params if p.ndim <= 1], "lr": HEAD_LR, "weight_decay": 0.0})
    return groups


optimizer = AdamW(build_param_groups())
total_steps = num_epochs * len(train_loader)
warmup_steps = int(0.1 * total_steps)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
print(f"{total_steps} optimiser steps, {warmup_steps} warmup")


def evaluate(loader, desc="Validating", amp=True):
    """Returns (loss, MAE, R2) in the original target units."""
    model.eval()
    total_loss, n_batches = 0.0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for texts, graph_pair, labels in tqdm(loader, desc=desc, unit="batch"):
            labels = labels.to(device)
            targets = (labels - TARGET_MEAN) / TARGET_STD
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=amp):
                # reshape(-1), not squeeze(): squeeze() collapses a final batch of size 1
                # to a 0-d tensor, which broadcasts silently in the loss and then raises
                # in .extend(...) below.
                predictions = model(texts, graph_pair, tokenizer).reshape(-1)
            predictions = predictions.float()
            loss = criterion(predictions, targets)
            total_loss += loss.item()
            n_batches += 1
            all_preds.extend((predictions * TARGET_STD + TARGET_MEAN).cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())
    model.train()
    return (total_loss / max(n_batches, 1),
            mean_absolute_error(all_labels, all_preds),
            r2_score(all_labels, all_preds))


best_val_mae = float("inf")
best_epoch = -1
epochs_without_improvement = 0
history = []
model.train()

for epoch in range(num_epochs):
    train_sampler.set_epoch(epoch)   # reshuffle shard buffers each epoch
    total_loss, nan_batches, seen = 0.0, 0, 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}", unit="batch")

    for step, (texts, graph_pair, labels) in enumerate(progress_bar):
        labels = labels.to(device)
        targets = (labels - TARGET_MEAN) / TARGET_STD

        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            predictions = model(texts, graph_pair, tokenizer).reshape(-1)
        loss = criterion(predictions.float(), targets)

        optimizer.zero_grad(set_to_none=True)
        if torch.isnan(loss):
            nan_batches += 1
            scheduler.step()   # still advance, so the schedule stays aligned with total_steps
            print(f"NaN loss at epoch {epoch+1}, step {step} - skipping batch")
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        seen += 1
        progress_bar.set_postfix(loss=f"{loss.item():.4f}", avg_loss=f"{total_loss / max(seen, 1):.4f}")

    train_avg_loss = total_loss / max(seen, 1)
    val_loss, val_mae, val_r2 = evaluate(val_loader)
    history.append({"epoch": epoch + 1, "train_loss": train_avg_loss,
                    "val_loss": val_loss, "val_mae": val_mae, "val_r2": val_r2})
    print(f"Epoch {epoch + 1}: train_loss={train_avg_loss:.4f}, val_loss={val_loss:.4f}, "
          f"val_MAE={val_mae:.4f}, val_R2={val_r2:.4f}, nan_batches={nan_batches}")

    if val_mae < best_val_mae:
        best_val_mae, best_epoch = val_mae, epoch
        epochs_without_improvement = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch,
            "val_mae": val_mae,
            "split_signature": _sig,
            "target_mean": TARGET_MEAN,
            "target_std": TARGET_STD,
            "encoder_meta": enc_meta,
        }, "./Results/Checkpoint/best_model.pth")
        print(f"  New best val MAE: {best_val_mae:.4f} - checkpoint saved")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= EARLY_STOP_PATIENCE:
            print(f"Early stop: no val improvement for {EARLY_STOP_PATIENCE} epochs "
                  f"(best {best_val_mae:.4f} @ epoch {best_epoch + 1})")
            break

    if (epoch + 1) % 5 == 0:
        fname = datetime.datetime.now().strftime('%Y-%m-%d-%H%M')
        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch,
            "loss": train_avg_loss,
        }, f"./Results/Checkpoint/checkpoint_alignn_matscibert_{epoch+1}_{fname}.pth")
        print(f"  Periodic checkpoint saved for epoch {epoch + 1}")

pd.DataFrame(history).to_csv("./Results/training_history.csv", index=False)
print(f"Best val MAE {best_val_mae:.4f} at epoch {best_epoch + 1}")


In [ ]:
# Evaluate the *best-validation* checkpoint, not whatever weights the loop ended on.
# The previous version scored the final epoch's model, which with early stopping (or with
# the old num_epochs=100) is well past the optimum.
ckpt = torch.load("./Results/Checkpoint/best_model.pth", map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Loaded best checkpoint: epoch {ckpt['epoch'] + 1}, val MAE {ckpt['val_mae']:.4f}, "
      f"split {ckpt['split_signature']}")
assert ckpt["split_signature"] == _sig, "checkpoint was trained on a different split"

# fp32 (amp=False): bf16 has an 8-bit mantissa, which is coarse enough to perturb a
# reported MAE at the third decimal place.
test_loss, test_mae, test_r2 = evaluate(test_loader, desc="Testing", amp=False)

print(f"Test SmoothL1 Loss: {test_loss:.4f}")
print(f"Test MAE: {test_mae:.4f}    (paper, band gap, CGCNN+SciBERT: 0.31 eV)")
print(f"Test R2:  {test_r2:.4f}     (paper, band gap: 0.79)")


## Known deviations from the paper (state these when reporting)

1. **Encoders swapped.** CGCNN -> ALIGNN, SciBERT -> MatSciBERT. The paper's own ablation
   (figure 6) finds MatSciBERT best on formation energy, so the text swap is supported;
   the graph swap is not something the paper evaluated (figure 7 covers GCN / SchNet /
   MEGNet / CGCNN only).

2. **The cross attention is degenerate.** Both embeddings are pooled to a single vector
   before `ImprovedAttentionCombiner`, so the softmax is over one element and is
   identically 1.0. The module is a linear map of the structure embedding plus a
   residual. This is inherited from the paper's implementation. Do not report attention
   weights as interpretable evidence.

3. **Text truncated at 512 tokens.** Mean description length is 741 words (table A2), so
   most Robocrystallographer descriptions are cut roughly in half.

4. **Targets are not standardised** (`NORMALIZE_TARGETS = False`), matching the paper.

5. **Dataset size.** The paper uses 95,582 Materials Project crystals; this run uses
   whatever survives the text + target + graph intersection printed above.

## Baselines still needed

Table 1's claim is relative (MatMMFuse 0.31 vs CGCNN 0.37 vs SciBERT 0.38). To make the
equivalent claim here you need, **on this exact split**:

- ALIGNN alone (already have it: `Band_ALIGNN_Encoder.ipynb`, val MAE ~0.265)
- MatSciBERT alone (text -> mean-pool -> linear head, same schedule)

Note the standalone ALIGNN already beats the paper's fused number, so the fusion result
has to be compared against it directly rather than against the paper's CGCNN baseline.
